# XAI-SkinNet - Kaggle one-click training
**Before running (one time):**
1. Right side -> **Session options**: Accelerator = **GPU T4 x2 or P100**, **Internet = On** (needed once to download pretrained weights).
2. **Add Input** -> search `skin-cancer-mnist-ham10000` (kmader) -> Add.
3. **Add Input** -> upload `xai-skinnet-main-kaggle.zip` as a new dataset (name it e.g. `xai-skinnet-code`) -> Add.
4. Top right: **Save Version -> Save & Run All (Commit)**. You can close the tab. Results appear under the notebook's **Output** tab (best.pt, history.json, metrics...).

In [ ]:
import os, sys, glob, zipfile, subprocess, shutil
from pathlib import Path

# --- find the project code (zip or already-unzipped) ---
WORK = Path('/kaggle/working/proj'); WORK.mkdir(exist_ok=True)
root = None
for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
    if 'skinnet' in z.lower():
        with zipfile.ZipFile(z) as zf: zf.extractall(WORK)
        break
for tr in glob.glob(str(WORK/'**/src/train.py'), recursive=True) + glob.glob('/kaggle/input/**/src/train.py', recursive=True):
    root = Path(tr).parents[1]; break
assert root, 'Project code not found - add the xai-skinnet zip as an input dataset'
if str(root).startswith('/kaggle/input'):
    shutil.copytree(root, WORK/'code', dirs_exist_ok=True); root = WORK/'code'
os.chdir(root); print('Project:', root)

# --- find HAM10000 ---
csvs = glob.glob('/kaggle/input/**/HAM10000_metadata.csv', recursive=True)
assert csvs, 'HAM10000 not found - add the kmader/skin-cancer-mnist-ham10000 input'
DATA_ROOT = str(Path(csvs[0]).parent); print('Data:', DATA_ROOT)
print('jpgs:', len(glob.glob(DATA_ROOT + '/**/*.jpg', recursive=True)))

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'Turn on a GPU in Session options and rerun'
!pip install -q 'timm>=0.9.12' grad-cam shap seaborn
# pretrained weights need internet; fail early with a clear message
import timm
timm.create_model('efficientnet_b4', pretrained=True); timm.create_model('vit_base_patch16_224', pretrained=True)
print('pretrained weights OK')

In [ ]:
import yaml
cfg = yaml.safe_load(open('configs/config.yaml'))
cfg['run_name'] = 'xai_skinnet_v2_meta'
cfg['train'].update(epochs=60, batch_size=32, freeze_epochs=5, early_stop_patience=12, num_workers=4,
                    imbalance='sampler', label_smoothing=0.1, backbone_lr_mult=0.1, select_metric='macro_f1')
cfg['model']['use_metadata'] = True
yaml.dump(cfg, open('configs/config.yaml','w'), default_flow_style=False)
print(yaml.dump(cfg['train']))

In [ ]:
# TRAIN v2 (about 2-3 hours). Saves last.pt every epoch to /kaggle/working/runs and auto-resumes if rerun.
!python -m src.train --config configs/config.yaml --data-root {DATA_ROOT} --output-dir /kaggle/working/runs --amp

In [ ]:
RUN = 'xai_skinnet_v2_meta'
CKPT = f'/kaggle/working/runs/{RUN}/best.pt'
!python -m src.evaluate --config configs/config.yaml --checkpoint {CKPT} --data-root {DATA_ROOT}
!python -m src.calibrate --checkpoint {CKPT} --data-root {DATA_ROOT}
print('--- calibrated, with flip/rotate test-time augmentation ---')
!python -m src.evaluate --config configs/config.yaml --checkpoint {CKPT} --data-root {DATA_ROOT} --tta

In [ ]:
import json, matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
h = json.load(open('/kaggle/working/runs/xai_skinnet_v2_meta/history.json'))
e=[x['epoch'] for x in h]
fig,(a,b)=plt.subplots(1,2,figsize=(13,4))
a.plot(e,[x['train_loss'] for x in h],label='train'); a.plot(e,[x['val_loss'] for x in h],label='val'); a.set_title('Loss'); a.legend()
b.plot(e,[x['train_acc'] for x in h],label='train'); b.plot(e,[x['val_acc'] for x in h],label='val'); b.set_title('Accuracy'); b.legend()
plt.savefig('/kaggle/working/runs/xai_skinnet_v2_meta/training_curves.png', dpi=120); print('best val_acc', max(x['val_acc'] for x in h))